# Module 2 — Building the End-to-End System with LangGraph

> **Time:** 25 minutes.
>
> **What you'll build:** the 4-agent Customer Support Triage system, end-to-end, runnable on a real ticket by the time you're done.

The design from Module 1 (state schema + topology) becomes runnable code in this notebook. We'll:

1. Install + set up
2. Define the typed `TriageState`
3. Build each agent — Classifier, Retriever, Drafter, QA
4. Wire them into a `StateGraph` with a feedback edge and a termination guard
5. Run it end-to-end on a billing ticket

**Definition of done:** `app.invoke(...)` returns a drafted reply for a sample ticket.

## 1.  Install + setup

Same dependencies as Module 0. If you already have them installed in this Colab session, the install is a no-op.

In [ ]:
%pip install -q \
    langgraph==0.2.* \
    langchain==0.3.* \
    langchain-openai==0.2.* \
    pydantic==2.* \
    openai==1.*


import os

def _ensure_key(name: str, optional: bool = False) -> None:
    """Load an API key from (in order): existing env, Colab Secrets, or getpass.

    Colab Secrets are the recommended path for this workshop — set them ONCE
    via the 🔑 key icon in Colab's left sidebar and every notebook will pick
    them up automatically.
    """
    if os.environ.get(name):
        print(f"  ✓  {name} already set in environment")
        return
    # 1. Try Colab Secrets
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            os.environ[name] = val
            print(f"  ✓  {name} loaded from Colab Secrets")
            return
    except Exception:
        pass
    # 2. Fallback — prompt the user
    import getpass
    val = getpass.getpass(f"Paste your {name}{' (optional)' if optional else ''}: ").strip()
    if val:
        os.environ[name] = val
        print(f"  ✓  {name} set")
    elif optional:
        print(f"  •  {name} skipped (optional)")
    else:
        print(f"  ⚠   {name} skipped — you'll hit errors later without it")

_ensure_key("OPENAI_API_KEY")

print("Ready.")


## 2.  The knowledge base (inline)

In production this would be a vector store. For the workshop we use a small in-memory dict keyed by category — deterministic, free, fast.

Each article has `id`, `title`, `text`, and `tags`. The Retriever just looks up by category and returns the matching articles.

In [ ]:
KB = {
    "billing": [
        {"id": "B-001",
         "title": "Billing cycle and prorations",
         "text": ("We bill on the same day each month. If you change plans mid-cycle, "
                  "the next invoice is prorated. Higher-than-usual charges are most "
                  "often a proration or a plan upgrade from the prior month.")},
        {"id": "B-002",
         "title": "Refund policy",
         "text": ("Refunds are available within 14 days of the original charge. "
                  "Refunds are issued to the original payment method and typically "
                  "settle within 5 business days.")},
        {"id": "B-003",
         "title": "Failed payments and retries",
         "text": ("If a payment fails, we retry once a day for 3 days. After that "
                  "the account is moved to a 7-day grace period.")},
    ],
    "technical": [
        {"id": "T-001",
         "title": "Login and authentication issues",
         "text": ("Most login failures resolve by clearing cookies for the domain. "
                  "If MFA isn't arriving, check spam and verify the registered phone.")},
        {"id": "T-002",
         "title": "API rate limits and 429 errors",
         "text": ("API rate limits are 60 requests per minute on Standard, 600/min "
                  "on Enterprise. 429 responses include a Retry-After header — "
                  "respect it with exponential backoff.")},
        {"id": "T-003",
         "title": "Data export failures",
         "text": ("Exports over 1 GB are split into multiple parts. If an export "
                  "shows 'failed', retry from the same dialog — the job is idempotent.")},
    ],
    "account": [
        {"id": "A-001",
         "title": "Resetting your password",
         "text": ("Use the 'Forgot password' link on the sign-in page. The reset "
                  "email is valid for 30 minutes. If you don't receive it, check "
                  "spam and confirm the address matches the registered account.")},
        {"id": "A-002",
         "title": "Closing or deleting an account",
         "text": ("Account closure is initiated from Settings → Account → Close "
                  "account. The account enters a 30-day pending-deletion window "
                  "during which it can be restored.")},
        {"id": "A-003",
         "title": "Transferring account ownership",
         "text": ("Ownership transfer requires the current owner to invite the new "
                  "owner as an Admin first, then both parties confirm via email.")},
    ],
}

# Quick sanity check
for cat, articles in KB.items():
    print(f"  {cat:10s}  {len(articles)} articles")

## 3.  The state schema

From Module 1's design. `revisions` uses the `Annotated[list, add]` reducer so each new revision is *appended* across loop iterations.

In [ ]:
from typing import TypedDict, Annotated, Literal
from operator import add


class TriageState(TypedDict):
    ticket: str
    category: Literal["billing", "technical", "account"] | None
    urgency:  Literal["low", "med", "high"] | None
    retrieved: list[dict]
    draft: str
    verdict: Literal["pass", "revise"] | None
    revision_count: int
    revisions: Annotated[list[str], add]


def make_initial_state(ticket: str) -> TriageState:
    """Build a fresh state for a new ticket."""
    return {
        "ticket": ticket,
        "category": None,
        "urgency": None,
        "retrieved": [],
        "draft": "",
        "verdict": None,
        "revision_count": 0,
        "revisions": [],
    }


print(make_initial_state("Why is my bill so high?"))

## 4.  LLM instance

We use **`gpt-4o-mini`** throughout — cheap, fast, plenty capable for this case study. `temperature=0` so reruns are reproducible (Module 3 will thank us for this).

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Smoke test
reply = llm.invoke("Reply 'pong' in lowercase.")
print(reply.content)

## 5.  Agent 1 — Classifier

In [ ]:
CLASSIFY_PROMPT = """\
You are a support ticket classifier.

Categorize the ticket into ONE of: billing, technical, account.
Rate urgency as ONE of: low, med, high.

High urgency = customer mentions a deadline, demo, outage, or repeated failures.
Medium = standard issue affecting work.
Low = informational question.

Return JSON only, no commentary:
{{"category": "...", "urgency": "..."}}

TICKET:
{ticket}
"""

In [ ]:
import json

def parse_json(text: str) -> dict:
    """Tolerant JSON parser — handles models that occasionally wrap output in ```."""
    text = text.strip()
    if text.startswith("```"):
        text = text.split("```")[1]
        if text.startswith("json"):
            text = text[4:]
    return json.loads(text.strip())


def classify(state: TriageState) -> dict:
    """Read the ticket. Decide category + urgency. Return ONLY the keys this node sets."""
    # TODO 1 — format CLASSIFY_PROMPT with the ticket
    prompt = ...

    # TODO 2 — call the LLM
    response = ...

    # TODO 3 — parse the JSON response
    parsed = ...

    # TODO 4 — return a partial state update with category and urgency
    return {
        # ...
    }


# Quick test — should print billing / med (or close)
result = classify(make_initial_state("My bill jumped by $40 this month, please explain."))
print(result)

## 6.  Agent 2 — Retriever

In [ ]:
def retrieve(state: TriageState) -> dict:
    """Look up KB articles for the classified category."""
    # TODO 1 — read the category from state
    category = ...

    # TODO 2 — fetch articles from KB (use KB.get to default to [] if unknown)
    docs = ...

    # TODO 3 — return a partial state update
    return {
        # ...
    }


# Test — feed a state that's already been through classify()
test_state = make_initial_state("My bill jumped this month.")
test_state["category"] = "billing"
print(retrieve(test_state))

## 7.  Agent 3 — Drafter

In [ ]:
DRAFT_PROMPT = """\
You are a customer support agent.

Reply to the ticket using ONLY the policies below. If a policy doesn't \
answer the question, say so honestly — do not invent details.

POLICIES:
{context}

TICKET:
{ticket}

Write a concise, helpful reply (3-6 sentences). Don't sign off with a name.
"""

In [ ]:
def draft(state: TriageState) -> dict:
    """Write a draft reply grounded in retrieved policies."""
    # TODO 1 — build the `context` string by joining the retrieved policies.
    # Each retrieved doc has 'id', 'title', 'text' — include the title and text.
    context = ...

    # TODO 2 — format DRAFT_PROMPT and call the LLM
    response = ...

    # TODO 3 — capture the draft in state.
    # IMPORTANT: also append it to `revisions` (the reducer makes this an append).
    draft_text = response.content
    return {
        # ...
    }


# Test
test_state = make_initial_state("My bill jumped by $40 this month, please explain.")
test_state["category"] = "billing"
test_state["retrieved"] = KB["billing"]
result = draft(test_state)
print(result["draft"])

## 8.  Agent 4 — QA Reviewer

In [ ]:
QA_PROMPT = """\
You are a quality assurance reviewer for a customer support team.

Decide whether the DRAFT below is acceptable to send to the customer.

Criteria for PASS:
- Addresses the ticket directly
- Uses information consistent with the policies provided
- Tone is professional and helpful

Otherwise, return REVISE.

Return JSON only:
{{"verdict": "pass" | "revise", "reason": "..."}}

TICKET:
{ticket}

POLICIES USED:
{context}

DRAFT:
{draft}
"""

In [ ]:
def qa(state: TriageState) -> dict:
    """Review the draft. Decide pass or revise. Increment revision_count."""
    # TODO 1 — build the context string (same shape as in draft())
    context = ...

    # TODO 2 — format QA_PROMPT, call LLM, parse JSON
    response = ...
    parsed = ...

    # TODO 3 — return the verdict AND increment revision_count by 1
    return {
        # "verdict": ...,
        # "revision_count": ...,
    }


# Test — feed a state with a draft
test_state = make_initial_state("My bill jumped by $40, please explain.")
test_state.update({
    "category": "billing",
    "retrieved": KB["billing"],
    "draft": "Your bill is high because of prorations from a recent plan change.",
})
print(qa(test_state))

## 9.  Wire the graph

In [ ]:
from langgraph.graph import StateGraph, END

MAX_REVISIONS = 2


def route_qa(state: TriageState) -> str:
    """Conditional edge after QA — pass to END, revise to drafter (bounded)."""
    # TODO 1 — if verdict is 'pass', go to END
    # TODO 2 — if revision_count is already at the cap, go to END (termination guard)
    # TODO 3 — otherwise, route back to 'drafter' for another attempt
    pass


# Build the graph
graph = StateGraph(TriageState)

# TODO 4 — register each agent as a node
# graph.add_node("classify", classify)
# ...

# TODO 5 — set the entry point
# graph.set_entry_point(...)

# TODO 6 — add the three static edges (classify → retrieve → drafter → qa)
# graph.add_edge(...)
# ...

# TODO 7 — add the conditional edge from qa using route_qa
# graph.add_conditional_edges("qa", route_qa, {"drafter": "drafter", "END": END})

# Compile
app = graph.compile()
print("Graph compiled.")

## 9.  Run the full pipeline

Invoke the graph on a sample ticket. The full state at the end shows what each node contributed.

In [ ]:
result = app.invoke(make_initial_state(
    "Hi, my monthly bill is $50 higher than last month and I didn't change anything. "
    "Can you check what happened?"
))

print(f"category        :  {result['category']}")
print(f"urgency         :  {result['urgency']}")
print(f"retrieved       :  {[d['id'] for d in result['retrieved']]}")
print(f"verdict         :  {result['verdict']}")
print(f"revision_count  :  {result['revision_count']}")
print()
print("DRAFT:")
print(result["draft"])

## 10.  Watch each node fire — `stream()`

For debugging (Module 3 territory), it's useful to see each node's update as it happens. This is your "trace by hand" tool.

In [ ]:
for step in app.stream(make_initial_state(
    "I want to close my account. Will everything really be deleted?"
)):
    for node, update in step.items():
        print(f"\n→  {node}")
        for k, v in update.items():
            preview = repr(v)
            if len(preview) > 80:
                preview = preview[:77] + "..."
            print(f"     {k} = {preview}")

## Wrap up

You shipped your first multi-agent system today. Recap of what's now wired up:

- ✅  Typed `TriageState` with a reducer for revisions
- ✅  Four agents, each a function returning a partial state update
- ✅  Sequential pipeline with a conditional feedback edge
- ✅  Termination guard (`max_revisions = 2`) preventing infinite loops
- ✅  Both `invoke()` and `stream()` working end-to-end

**Up next:** Module 3 — Debugging & Failure Modes. We'll break this system on purpose and learn to spot what went wrong.